In [ ]:
from pathlib import Path
import sys, os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import neuropythy as ny
import immlib as il

In [ ]:
# Make sure the drawFAVA src directory is on the path.
srcpath = Path() / 'drawFAVA-cortex-annotate' / 'src'
srcpath = os.fspath(srcpath.resolve())
if srcpath not in sys.path:
    sys.path.append(srcpath)
    
import annotate as ann

## Load in the annotations themselves.

<div class="alert alert-block alert-info">
<b>Note:</b>

This notebook is meant to be run within the root of the data branch of the nbenlab/drawFAVA github repository. This branch integrates the annotations of the project's research assistants, each of whom annotated the datasets in their own forks of the nbenlab/drawFAVA repository.
    
The subdirectories of the annotation/ directory can be refreshed (i.e., reintegrated by merging from the downstream forks) using the merge-forks.sh script in the root of the nbenlab/drawFAVA:data branch.

</div>

In [ ]:
drawfava_path = Path("drawFAVA")
annotations_path = drawfava_path / "annotations"

annotators = [
    dict(github='mahis1026',       name='Mahi Shah'),
    dict(github='j2bee',           name='John Bee'),
    dict(github='ZariaP',          name='Zaria Piao'),
    dict(github='tsukhee',         name='Nisha'),
    dict(github='sarora2856',      name='Sagarika Arora'),
    dict(github='longhinm09',      name='Martina Longhin'),
    dict(github='SuchithaJ',       name='Suchitha Jagalur'),
    dict(github='idhrcontrol',     name='Idhant Rao'),
    dict(github='ikaplun-glitch',  name='Isabella Kaplun'),
    dict(github='sidikhaliya-del', name='Khaliya Sidi'),
    dict(github='shruthikab',      name='Shruthika Balasubramanian'),
    dict(github='ziyil115',        name='Ziyi Liu')]

In [ ]:
def loadall(path=annotations_path):
    path = Path(path)
    r = dict(
        dataset=[], annotator=[], subject=[],
        hemisphere=[], annotation=[], points=[])
    for dset in loadall.datasets:
        dpath = path / dset
        if not dpath.is_dir():
            print("Dataset not found:", dset)
            continue
        for annot in annotators:
            gh = annot['github']
            apath = dpath / gh
            if not apath.is_dir():
                continue
            for spath in apath.iterdir():
                if spath.name[0] == '.' or not spath.is_dir():
                    continue
                for hpath in spath.iterdir():
                    if hpath.name == 'Right Hemisphere':
                        h = 'rh'
                    elif hpath.name == 'Left Hemisphere':
                        h = 'lh'
                    else:
                        continue
                    for fl in hpath.iterdir():
                        if not fl.name.endswith('.tsv'):
                            continue
                        r['dataset'].append(dset)
                        r['annotator'].append(gh)
                        r['subject'].append(spath.name)
                        r['hemisphere'].append(h)
                        r['annotation'].append(fl.name[:-4])
                        r['points'].append(pd.read_csv(fl, sep='\t').values)
    return pd.DataFrame(r)
loadall.datasets = ('hcp-retinotopy',)

In [ ]:
df = loadall()
df

In [ ]:
summary_df = {'annotator':[], 'subject':[], 'npoints':[], 'nannotations':[]};

for gh in np.unique(df['annotator']):
    df1 = df[df['annotator'] == gh]
    for (name, group) in df1.groupby(by='subject'):
        summary_df['annotator'].append(gh)
        summary_df['subject'].append(name)
        npts = 0
        ncon = 0
        for el in group['points']:
            npts += len(el)
            ncon += 1
        summary_df['npoints'].append(npts)
        summary_df['nannotations'].append(ncon)

summary_df = pd.DataFrame(summary_df)
summary_df

In [ ]:
rows = [ann['github'] for ann in annotators]
cols = np.unique(summary_df['subject'])

mtx = [
    [np.sum(sdf['npoints'])
     for sid in cols
     for sdf in [adf[adf['subject'] == sid]]]
    for ann in rows
    for adf in [summary_df[summary_df['annotator'] == ann]]]

mtx = np.array(mtx)

In [ ]:
(fig,ax) = plt.subplots(1, 1, figsize=(5,4), dpi=144)

ax.imshow(mtx, cmap='gray')
ax.set_yticks(range(len(mtx)))
ax.set_yticklabels(rows)
ax.set_xticks(range(len(mtx[0])))
ax.set_xticklabels(cols)

pass

## Code for Exporting Images/Books of Annotations by Rater

### Code to Run in the Annotation Tool Docker/Notebook

First, check out the `nbenlab/drawFAVA` repository and switch to the `data` branch. Go to the `annotations` directory, and in every dataset directory, make a symbolic link between any one of the rater subdirectories to `nbenlab`; e.g.:

```bash
for dir in annotations/*
do { cd ${dir} && ln -s ./j2bee/ ./nbenlab; }
done
```

Once you have done this, go back to the root of the repository and start the annotation tool via `docker compose up`. 
The tool should load as normal. While in the notebook, you can press escape then `b` to create a new input cell. Copy-and-paste the following into that cell in order to disable the input cell hiding throughout the notebook.

```html
%%html

<script>
        $('div.prompt').show();
        $('div.input').show();
</script>
```

Following the above, it should be relatively straightforward to use the notebook normally. You can use the following code-block to define an asynchronous function that saves images of each set of annotations drawn by a given rater or raters.

```python
all_datasets = ('hcp-retinotopy',)
all_raters = (
    'idhrcontrol', 'j2bee', 'longhinm09', 'shruthikab', 'SuchithaJ',
    'ZariaP', 'ikaplun-glitch', 'mahis1026', 'sarora2856',
    'sidikhaliya-del', 'tsukhee', 'ziyil115')
async def save_multicanvas(mc0, filename, delay=1.5, handle=None):
    import asyncio
    from IPython.display import display
    from ipycanvas import MultiCanvas
    mc = MultiCanvas(
        n_canvases=len(mc0._canvases),
        width=mc0.width,
        height=mc0.height,
        sync_image_data=True)
    def _save_to_file(*args, **kwargs):
        mc.to_file(filename)
    mc.observe(_save_to_file, "image_data")
    for ii in range(len(mc0._canvases)):
        mc[ii].draw_image(mc0[ii], 0, 0)
    # Display temporarily
    if handle is None:
        handle = display(mc, display_id=True)
    else:
        handle.update(mc)
    # Give frontend time to render + sync image data
    await asyncio.sleep(delay)
    # Save
    return handle    
async def save_images(datasets=None, raters=None, *, 
                      outdir='/save',
                      dataset_names=datasets,
                      annotation_tools=annotation_tools):
    from glob import glob
    from pathlib import Path
    if raters is None:
        raters = all_raters
    if datasets is None:
        datasets = all_datasets
    handle = None
    tophand = display("", display_id=True)
    for title in datasets:
        nbenlab_dir = Path(f"/save/{title}/nbenlab")
        for rater in raters:
            rater_dir = Path(f"/save/{title}/{rater}")
            if not rater_dir.is_dir():
                print(f"*** No rater {rater} for dataset {title}")
                continue
            nbenlab_dir.unlink(missing_ok=True)
            nbenlab_dir.symlink_to(rater_dir)
            subdirs = list(map(Path, glob(str(nbenlab_dir / "*"))))
            sids = [sd.name for sd in subdirs if sd.is_dir()]
            # Now taht we've relinked the directory, remake the tool.
            tool = AnnotationTool(
                config_path = f"/config/{title}.yaml", 
                cache_path  = f"/cache/{title}", 
                save_path   = f"/save/{title}")
            tool.control_panel.figure_size_slider.value = 425
            tool.sync_image_data = True
            mc = tool.children[1].children[1]
            mc.sync_image_data = True

            sid_select = tool.control_panel.selection_panel.children[1]
            hem_select = tool.control_panel.selection_panel.children[2]
    
            nhemis = len(sids) * 2
            iihem = 0
            for sid in sids:
                sid_select.index = sid_select.options.index(sid)
                for h in ('Left Hemisphere', 'Right Hemisphere'):
                    hstr = h[0].lower() + 'h'
                    filename = nbenlab_dir / f"{sid}.{hstr}.png"
                    if filename.is_file():
                        continue
                    hem_select.index = hem_select.options.index(h)
                    handle = await save_multicanvas(mc, str(filename), handle=handle)
                    iihem += 1
                    tophand.update(f"{title}: {rater};  {iihem} of {nhemis} hemispheres done.")
    handle.update("Finished!")
```

Then, one at a time, you can run the following in a cell.

```python
#await save_images(raters=['idhrcontrol'])
#await save_images(raters=['longhinm09'])
#await save_images(raters=['shruthikab'])
#await save_images(raters=['SuchithaJ'])
#await save_images(raters=['ZariaP'])
#await save_images(raters=['sarora2856'])
#await save_images(raters=['sidikhaliya-del'])
#await save_images(raters=['tsukhee'])
#await save_images(raters=['ziyil115'])
#await save_images(raters=['j2bee'])
#await save_images(raters=['ikaplun-glitch'])
#await save_images(raters=['mahis1026'])
```

Unfortunately, if you queue them all up at once, the images don't save out until they all finish, which may cause an out-of-memory crash.

### To Run in this Notebook after Generatng PNG files

Once you've generated png files for all the annotators using the code above, you can collate them into PDF files using the code below.

In [ ]:
%%bash

REPORTS_DIR="${HOME}/Desktop/annot-reports"
ANNOT_DATABRANCH_REPO_DIR="${HOME}/Code/drawFAVA-data/work/annotation/drawFAVA"
DATASETS=('hcp-retinotopy')

set -eo pipefail

for DS in "${DATASETS[@]}"
do cd "${ANNOT_DATABRANCH_REPO_DIR}/annotations/${DS}"
   mkdir -p "${REPORTS_DIR}/pngs/${DS}"
   mkdir -p "${REPORTS_DIR}/pdfs/${DS}"
   for rater in *
   do if ! [ -s "${rater}" ]
      then continue
      fi
      if [ -r "${REPORTS_DIR}/pdfs/${DS}/${rater}".pdf ]
      then continue
      fi
      mkdir -p "${REPORTS_DIR}/pngs/${DS}/${rater}"
      for flnm in $(find ./$rater/ -name '*.png')
      do f=$(basename $flnm)
         sid="${f:0:6}"
         h="${f:7:2}"
         magick "${flnm}" \
                -resize 1024 \
                -font /System/Library/Fonts/HelveticaNeue.ttc -pointsize 24 -fill black \
                -gravity center -annotate +0-50 \
                "${sid} ${h}" \
                "${REPORTS_DIR}/pngs/${DS}/${rater}/${f}"
         rm ${flnm}
      done
      # Make a PDF file of their annotations.
      magick "${REPORTS_DIR}/pngs/${DS}/${rater}"/*.png \
             "${REPORTS_DIR}/pdfs/${DS}/${rater}".pdf
   done
done